# 03 Piecewise Interpolation and Cubic Splines

Piecewise methods avoid forcing one high-degree polynomial through all data
points. They build local approximations on subintervals, which usually improves
robustness and makes the effect of changing one node more local.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import NaturalCubicSpline, lagrange_interpolate, piecewise_linear_interpolate


## Piecewise linear interpolation

On each interval $[x_i,x_{i+1}]$, piecewise linear interpolation uses

$$
p(x)=\frac{x_{i+1}-x}{x_{i+1}-x_i}y_i+
\frac{x-x_i}{x_{i+1}-x_i}y_{i+1}.
$$

It is local and simple, but the derivative is discontinuous at the nodes.


In [ ]:
x = np.array([0.0, 0.8, 1.7, 2.5, 3.2, 4.0])
y = np.cos(1.5 * x) + 0.2 * x
x_eval = np.linspace(x.min(), x.max(), 500)

linear = piecewise_linear_interpolate(x, y, x_eval)
polynomial = lagrange_interpolate(x, y, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, polynomial, label="Global polynomial")
plt.plot(x_eval, linear, label="Piecewise linear")
plt.scatter(x, y, color="black", zorder=3, label="Data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Global versus local interpolation")
plt.legend()
plt.tight_layout()
plt.show()


## Locality experiment

Changing one data value changes a global polynomial everywhere. A piecewise
linear interpolant changes only on the neighboring intervals.


In [ ]:
y_perturbed = y.copy()
y_perturbed[3] += 0.8

poly_perturbed = lagrange_interpolate(x, y_perturbed, x_eval)
linear_perturbed = piecewise_linear_interpolate(x, y_perturbed, x_eval)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(x_eval, polynomial, label="Original")
axes[0].plot(x_eval, poly_perturbed, label="Perturbed")
axes[0].scatter(x, y_perturbed, color="black", s=20)
axes[0].set_title("Global polynomial")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

axes[1].plot(x_eval, linear, label="Original")
axes[1].plot(x_eval, linear_perturbed, label="Perturbed")
axes[1].scatter(x, y_perturbed, color="black", s=20)
axes[1].set_title("Piecewise linear")
axes[1].set_xlabel("x")
axes[1].legend()

fig.suptitle("Local methods limit the spread of a local data change")
fig.tight_layout()
plt.show()


## Natural cubic spline

A cubic spline uses one cubic polynomial per interval,

$$
S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3,
$$

and enforces continuity of $S$, $S'$, and $S''$ at interior nodes. The natural
boundary condition is

$$
S''(x_0)=S''(x_n)=0.
$$

The resulting linear system is tridiagonal and can be solved efficiently.


In [ ]:
spline = NaturalCubicSpline.fit(x, y)
spline_values = spline(x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, linear, label="Piecewise linear")
plt.plot(x_eval, spline_values, label="Natural cubic spline")
plt.scatter(x, y, color="black", zorder=3, label="Data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Spline interpolation adds smoothness")
plt.legend()
plt.tight_layout()
plt.show()

print("interval coefficients")
for i, (a, b, c, d) in enumerate(zip(spline.a, spline.b, spline.c, spline.d)):
    print(f"i={i}: a={a: .4f}, b={b: .4f}, c={c: .4f}, d={d: .4f}")


## Boundary-condition roadmap

This first implementation uses natural endpoint conditions. A later pass should
add clamped conditions, where endpoint derivatives are specified, and compare
how endpoint behavior changes.


## Summary

* Piecewise linear interpolation is robust and local but not smooth.
* Natural cubic splines keep locality while enforcing smooth first and second derivatives.
* The spline system is tridiagonal, which connects this chapter to later linear-system methods.
